# Maia Corpus v9 — Rights Recovery v0.10
Hugging Face First. Não altera os livros existentes. Recupera metadados em lote; candidatos não equivalem a autorização jurídica.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import csv,json,re,os,hashlib,collections,time
from huggingface_hub import HfApi,hf_hub_download
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_4_0/maia-corpus-v9')
OUT=ROOT/'gutenberg_v10_hf_rights'; OUT.mkdir(exist_ok=True)
CACHE=OUT/'cache'; CACHE.mkdir(exist_ok=True)
REPOS=['Texttechnologylab/project-gutenberg-temporal-corpus','zkeown/gutenberg-corpus','sedthh/gutenberg_english','sedthh/gutenberg_multilang','common-pile/project_gutenberg_filtered']
def atomic(p,obj):
    t=p.with_suffix('.part'); t.write_text(json.dumps(obj,ensure_ascii=False,default=str)); os.replace(t,p)
def norm(s):return re.sub('[^a-z0-9]','',str(s).lower())
def gid(v):
    s=str(v or '').strip()
    m=re.fullmatch(r'(?:pg)?(\d+)(?:-\d+)?(?:\.txt)?',s,re.I) or re.search(r'gutenberg\.org/(?:ebooks|files|cache/epub)/(\d+)(?:/|$)',s)
    return str(int(m.group(1))) if m else ''
with (ROOT/'gutenberg_v08/rights/rights_inventory_latest.csv').open(encoding='utf-8-sig') as f:
    items=list(csv.DictReader(f))
print('Inventário:',len(items), 'colunas:',list(items[0]))
ID={'id','bookid','gutenbergid','gutenbergnumericid','ebookid','ebook','textid','pgid','url','gutenbergurl','textnumber'}
def rid(r):
    for k,v in r.items():
        if norm(k) in ID or k=='Text#':
            x=gid(v)
            if x:return x
    if isinstance(r.get('metadata'),dict):return rid(r['metadata'])
    return ''
for r in items:r['_gid']=rid(r)
wanted={r['_gid'] for r in items if r['_gid']}
assert len(items)==39340 and sum(bool(r['_gid']) for r in items)>.95*len(items), 'Inventário ou IDs inesperados; não prosseguir.'
print('IDs reconhecidos:',len(wanted))

## Descoberta de fontes
Os repositórios são candidatos; disponibilidade e esquema serão verificados em execução. Fixamos cada fonte por commit.

In [ ]:
api=HfApi(); sources=[]
for repo in REPOS:
    try:
        info=api.dataset_info(repo,files_metadata=True,timeout=30)
        files=[{'name':s.rfilename,'size':s.size} for s in info.siblings if s.rfilename.endswith(('.csv','.csv.gz','.jsonl','.parquet'))]
        sources.append({'repo':repo,'revision':info.sha,'files':files})
        print(repo,info.sha,len(files),[f['name'] for f in files[:5]],flush=True)
    except Exception as e:print('FALHA',repo,str(e)[:300],flush=True)
atomic(OUT/'discovery.json',sources)
assert sources,'Nenhuma fonte disponível' 

## Recuperação retomável
CSV de metadados primeiro; Parquet com projeção de colunas por HTTP range. Cache por arquivo e commit, sem baixar novamente os textos originais. Limite de 100 MB para CSV/JSONL e 300 arquivos Parquet por fonte. Falhas ficam registradas.

In [ ]:
import pyarrow.parquet as pq
import fsspec,aiohttp,gzip
META={'title','author','authors','creator','creators','authoryearofdeath','authordeathyear','authordeathyears','deathdate','deathyear','death','language','languages','rights','copyright','license','licence','translator','translators','editor','editors','contributors','contributor','issued','publicationdate','metadata','authorname','authornames'}
results=[]; events=[]
def process(src,file):
    repo,rev,name=src['repo'],src['revision'],file['name']
    key=hashlib.sha256(json.dumps([repo,rev,name,sorted(wanted)]).encode()).hexdigest()
    dest=CACHE/(key+'.json')
    if dest.exists():return json.loads(dest.read_text())
    matches={}; scanned=0; cols=[]
    def accept(r):
        nonlocal scanned
        scanned+=1; ident=rid(r)
        meta={k:v for k,v in r.items() if norm(k) in META}
        if ident in wanted and any(v for v in meta.values()):matches.setdefault(ident,[]).append(meta)
        if scanned%20000==0:print('  linhas',scanned,'matches',len(matches),flush=True)
    if name.endswith('.parquet'):
        url=f'https://huggingface.co/datasets/{repo}/resolve/{rev}/{name}'
        with fsspec.open(url,'rb',block_size=1024*1024,client_kwargs={'timeout':aiohttp.ClientTimeout(total=180,sock_read=45)}) as f:
            pf=pq.ParquetFile(f);cols=pf.schema_arrow.names
            selected=[k for k in cols if norm(k) in META|ID or k=='Text#']
            if not any(norm(k) in ID|{'metadata'} or k=='Text#' for k in selected) or not any(norm(k) in META for k in selected):raise ValueError('Esquema incompatível: '+str(cols))
            for batch in pf.iter_batches(batch_size=2048,columns=selected):
                for r in batch.to_pylist():accept(r)
    else:
        local=hf_hub_download(repo_id=repo,repo_type='dataset',filename=name,revision=rev,etag_timeout=30)
        opener=gzip.open if name.endswith('.gz') else open
        with opener(local,'rt',encoding='utf-8-sig',errors='replace') as f:
            if name.endswith(('.csv','.csv.gz')):
                reader=csv.DictReader(f);cols=reader.fieldnames or []
                for r in reader:accept(r)
            else:
                for line in f:
                    r=json.loads(line)
                    if isinstance(r,dict):accept(r)
    obj={'repo':repo,'revision':rev,'file':name,'columns':cols,'scanned':scanned,'matches':matches};atomic(dest,obj);return obj
for src in sources:
    files=sorted(src['files'],key=lambda f:(f['name'].endswith('.parquet'),f['size'] or 0)); count=0
    for f in files:
        if f['name'].endswith('.parquet'):
            count+=1
            if count>300:continue
        elif f['size'] is None or f['size']>100_000_000:continue
        print('LENDO',src['repo'],f['name'],flush=True)
        for attempt in range(2):
            try:
                obj=process(src,f);results.append(obj);events.append({'repo':src['repo'],'file':f['name'],'matches':len(obj['matches']),'status':'ok'});print('OK',len(obj['matches']),flush=True);break
            except Exception as e:
                print('FALHA',str(e)[:300],flush=True)
                if attempt==1:events.append({'repo':src['repo'],'file':f['name'],'status':'failed','error':str(e)[:500]})
                else:time.sleep(3)
        atomic(OUT/'progress.json',events)
    covered={k for o in results for k in o['matches']};print('Cobertura',len(covered),'/',len(wanted),flush=True)
    if covered==wanted:break
assert any(o['matches'] for o in results),'Zero metadados: interrompido antes da auditoria.' 

## Consolidar evidências
A data de morte até 1955 é apenas indício segundo a regra geral brasileira em 2026. Direitos de edição, tradução e coautoria não são inferidos. Nenhuma aprovação automática definitiva é implementada nesta versão.

In [ ]:
index=collections.defaultdict(list)
for o in results:
    for ident,meta in o['matches'].items():
        index[ident].append({'repo':o['repo'],'revision':o['revision'],'file':o['file'],'metadata':meta})
output=[]
for r in items:
    ev=index.get(r['_gid'],[]); years=[]
    for e in ev:
        for meta in e['metadata']:
            for k,v in meta.items():
                if norm(k) in {'authoryearofdeath','authordeathyear','authordeathyears','deathdate','deathyear','death'}:years.extend(int(y) for y in re.findall(r'(?<!\d)(?:1[0-9]{3}|20[0-9]{2})(?!\d)',str(v)))
    status='CANDIDATE_PUBLIC_DOMAIN' if years and max(years)<=1955 else 'AUTO_QUARANTINE'
    output.append({**r,'hf_metadata_found':bool(ev),'hf_death_years':';'.join(map(str,sorted(set(years)))),'hf_rights_status':status,'approved_for_training':False,'hf_evidence_json':json.dumps(ev,ensure_ascii=False,default=str)})
p=OUT/'rights_inventory_latest.csv';tmp=p.with_suffix('.part')
with tmp.open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(dict.fromkeys(k for r in output for k in r)));w.writeheader();w.writerows(output)
os.replace(tmp,p)
summary={'books':len(output),'metadata_matched':sum(r['hf_metadata_found'] for r in output),'status':dict(collections.Counter(r['hf_rights_status'] for r in output)),'approved':0,'by_language':{lang:{'books':sum(r.get('language')==lang for r in output),'matched':sum(r.get('language')==lang and r['hf_metadata_found'] for r in output)} for lang in ['pt','en','es']}}
atomic(OUT/'rights_summary_latest.json',summary);print(json.dumps(summary,indent=2));print(p)

Reexecute para retomar: arquivos concluídos são reutilizados. Envie rights_summary_latest.json e progress.json. Não há contagem de tokens nem limpeza nesta versão.